# fase_4 - script_afrida Migration

This notebook handles migration of database from old DB to new DB for fase 4.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [3]:
import pandas as pd

# =========================================================
# 1. EXTRACT DATA DARI DB LAMA
# =========================================================
print("📥 Mengambil data jadwal dari database lama...")
cursor_old.execute("SELECT * FROM jadwal")
data_jadwal_old = cursor_old.fetchall()
df_raw_jadwal = pd.DataFrame(data_jadwal_old)
print(f"Total data asli di DB lama: {len(df_raw_jadwal)} baris")


📥 Mengambil data jadwal dari database lama...
Total data asli di DB lama: 551 baris


In [4]:
# Extract tabel jadwal
df_jadwal_lama = pd.read_sql("SELECT * FROM jadwal", db_old)

In [5]:
display(df_jadwal_lama.head())

,idjadwal,groupwa,idsesi,idpendkursus,idperiode,hari,idlevel,idzoom,mode_belajar,tempat,status_archive
0,J000000023,01 GOGO 3B SR2 (ERICA),S00002,K00001,P00006,"Senin,Rabu",L00017,Z00004,Online,Ruang Kelas 4,1.0
1,J000000024,02 SO 1C SR2 (QORIN),S00002,K00001,P00006,"Senin,Rabu",L00024,Z00006,Offline,Ruang Kelas 5,1.0
2,J000000025,03 SO 1B SR1 (TATIK),S00001,K00001,P00006,"Senin,Rabu",L00023,Z00006,Offline,Ruang Kelas 1,1.0
3,J000000026,04 SO 2A SR3 (TATIK),S00003,K00001,P00006,"Senin,Rabu",L00025,Z00006,Offline,Ruang Kelas 1,1.0
4,J000000027,05 GOGO 1B SelK3 (ERICA),S00003,K00001,P00006,"Selasa,Kamis",L00014,Z00006,Offline,Ruang Kelas 4,1.0


In [19]:
import pickle

# =========================================================
# 2. TRANSFORMASI TABEL: jadwal
# =========================================================
print("⚡ Melakukan transformasi tabel 'jadwal'...")

# A. Pembersihan metode belajar (sesuai enum: Online, Offline, Hybrid)
def clean_mode_belajar(val):
    if pd.isna(val) or not str(val).strip():
        return 'Offline'
    s = str(val).strip().capitalize()
    if s in ('Online', 'Offline', 'Hybrid'):
        return s
    return 'Offline'

# B. Pembersihan status arsip (cast float ke integer tinyint 0 atau 1)
def clean_status_arsip(val):
    try:
        if pd.isna(val):
            return 0
        return int(float(val))
    except:
        return 0

# C. Saring (filter out) jadwal percobaan agar tidak melanggar Foreign Key
df_jadwal_clean = df_raw_jadwal[~df_raw_jadwal['idperiode'].isin(['P00094', 'P00104'])].copy()
skipped_count = len(df_raw_jadwal) - len(df_jadwal_clean)
print(f"ℹ️ Menyaring {skipped_count} data jadwal percobaan (sandbox). Sisa data: {len(df_jadwal_clean)} baris")


# =========================================================
# 3. TRANSFORMASI TABEL BARU: jadwal_hari
# =========================================================
print("⚡ Memisahkan kolom 'hari' ke tabel 'jadwal_hari'...")

hari_rows = []
for idx, row in df_jadwal_clean.iterrows():
    old_id_jadwal = row['idjadwal']
    hari_string = row['hari']
    
    if pd.isna(hari_string) or not str(hari_string).strip():
        continue
        
    hari_list = [h.strip() for h in hari_string.split(',') if h.strip()]
    for hari in hari_list:
        hari_rows.append({
            'id_jadwal': old_id_jadwal,  # Tetap simpan string ID lama
            'nama_hari': hari
        })

df_jadwal_hari = pd.DataFrame(hari_rows)
print(f"✓ Tabel 'jadwal_hari' siap. Shape: {df_jadwal_hari.shape}")

# ... sebelumnya sudah buat df_jadwal_clean

# Siapkan dataframe untuk insert (tanpa id_jadwal)
df_jadwal_insert = pd.DataFrame()
df_jadwal_insert['id_kursus'] = df_jadwal_clean['idpendkursus']
df_jadwal_insert['id_periode'] = df_jadwal_clean['idperiode']
df_jadwal_insert['id_level'] = df_jadwal_clean['idlevel']
df_jadwal_insert['id_sesi'] = df_jadwal_clean['idsesi']
df_jadwal_insert['metode_belajar_jadwal'] = df_jadwal_clean['mode_belajar'].apply(clean_mode_belajar)
df_jadwal_insert['nama_rombel'] = df_jadwal_clean['groupwa'].fillna('').str.strip()
df_jadwal_insert['status_arsip'] = df_jadwal_clean['status_archive'].apply(clean_status_arsip)
df_jadwal_insert['tempat'] = df_jadwal_clean['tempat'].fillna('').str.strip()

# Simpan urutan old_id_jadwal (penting untuk mapping nanti)
old_id_list = df_jadwal_clean['idjadwal'].tolist()  # urutan sama dengan df_jadwal_insert

# ... buat df_jadwal_hari (masih pakai old_id) seperti kode sebelumnya

# Kemudian kumpulkan ke dictionary
fase_4_afrida = {
    'jadwal': df_jadwal_insert,
    'jadwal_old_ids': old_id_list,
    'jadwal_hari': df_jadwal_hari,   # kolom 'id_jadwal' berisi old_id string
    # nanti tambahkan 'jadwal_detail', 'jadwal_pengajar', 'jadwal_siswa'
}

⚡ Melakukan transformasi tabel 'jadwal'...
ℹ️ Menyaring 2 data jadwal percobaan (sandbox). Sisa data: 549 baris
⚡ Memisahkan kolom 'hari' ke tabel 'jadwal_hari'...
✓ Tabel 'jadwal_hari' siap. Shape: (975, 2)


In [8]:
display(df_jadwal_insert.head())

,id_kursus,id_periode,id_level,id_sesi,metode_belajar_jadwal,nama_rombel,status_arsip,tempat
0,K00001,P00006,L00017,S00002,Online,01 GOGO 3B SR2 (ERICA),1,Ruang Kelas 4
1,K00001,P00006,L00024,S00002,Offline,02 SO 1C SR2 (QORIN),1,Ruang Kelas 5
2,K00001,P00006,L00023,S00001,Offline,03 SO 1B SR1 (TATIK),1,Ruang Kelas 1
3,K00001,P00006,L00025,S00003,Offline,04 SO 2A SR3 (TATIK),1,Ruang Kelas 1
4,K00001,P00006,L00014,S00003,Offline,05 GOGO 1B SelK3 (ERICA),1,Ruang Kelas 4


In [9]:
# =========================================================
# 4. TRANSFORMASI TABEL: jadwal_detail (sumber: jadwal_detil)
#    dengan asumsi id_jadwal_detail auto increment
# =========================================================
print("⚡ Melakukan transformasi tabel 'jadwal_detail'...")

df_detil_lama = pd.read_sql("SELECT * FROM jadwal_detil", db_old)
print(f"  Data mentah: {len(df_detil_lama)} baris")

# Filter: hanya jadwal yang sudah lolos filter (ada di valid_jadwal_ids)
valid_jadwal_ids = set(df_jadwal_clean['idjadwal'])
df_detil_lama = df_detil_lama[df_detil_lama['idjadwal'].isin(valid_jadwal_ids)]
print(f"  Setelah filter idjadwal: {len(df_detil_lama)} baris")

# Simpan old_id_jadwal_detail untuk mapping nanti (dari idjadwaldetil)
old_detail_ids = df_detil_lama['idjadwaldetil'].tolist()

# Dataframe untuk insert (tanpa id_jadwal_detail, karena auto increment)
df_jadwal_detail_insert = pd.DataFrame()
df_jadwal_detail_insert['judul'] = df_detil_lama['title'].fillna('').astype(str)
df_jadwal_detail_insert['deskripsi'] = df_detil_lama['description'].fillna('').astype(str)
df_jadwal_detail_insert['url_jadwal_detail'] = df_detil_lama['url'].fillna('').astype(str)
df_jadwal_detail_insert['id_jadwal'] = df_detil_lama['idjadwal']  # masih string old_id (akan di-map nanti)
df_jadwal_detail_insert['label_warna'] = df_detil_lama['color'].fillna('').astype(str)

# Konversi start/end ke date
df_jadwal_detail_insert['penanda_mulai'] = pd.to_datetime(df_detil_lama['start'], errors='coerce').dt.date
df_jadwal_detail_insert['penanda_selesai'] = pd.to_datetime(df_detil_lama['end'], errors='coerce').dt.date

# Kolom id_mitra (sementara NULL)
df_jadwal_detail_insert['id_mitra'] = None
df_jadwal_detail_insert['id_sesi_override'] = None
df_jadwal_detail_insert['status_detail'] = 'active'
df_jadwal_detail_insert['source_type'] = 'migrasi'
df_jadwal_detail_insert['original_jadwal_detail_id'] = None
df_jadwal_detail_insert['has_operational_data'] = 0
df_jadwal_detail_insert['last_generated_at'] = None
df_jadwal_detail_insert['created_at'] = pd.Timestamp.now()
df_jadwal_detail_insert['updated_at'] = pd.Timestamp.now()

print(f"✓ df_jadwal_detail_insert siap. Shape: {df_jadwal_detail_insert.shape}")

# Simpan ke dictionary
fase_4_afrida['jadwal_detail'] = df_jadwal_detail_insert
fase_4_afrida['jadwal_detail_old_ids'] = old_detail_ids

⚡ Melakukan transformasi tabel 'jadwal_detail'...
  Data mentah: 17267 baris
  Setelah filter idjadwal: 17257 baris
✓ df_jadwal_detail_insert siap. Shape: (17257, 16)


In [10]:
display(df_jadwal_detail_insert.head())

,judul,deskripsi,url_jadwal_detail,id_jadwal,label_warna,penanda_mulai,penanda_selesai,id_mitra,id_sesi_override,status_detail,source_type,original_jadwal_detail_id,has_operational_data,last_generated_at,created_at,updated_at
0,08 SO 2A SelK3 (GETA),,https://us02web.zoom.us/j/5403514055?pwd=ZnZ2d...,J000000030,fc-event-info,2023-07-04,2023-07-05,None,None,active,migrasi,None,0,None,2026-06-02 14:22:58.978733,2026-06-02 14:22:58.979526
1,08 SO 2A SelK3 (GETA),,https://us02web.zoom.us/j/5403514055?pwd=ZnZ2d...,J000000030,fc-event-info,2023-07-06,2023-07-07,None,None,active,migrasi,None,0,None,2026-06-02 14:22:58.978733,2026-06-02 14:22:58.979526
2,08 SO 2A SelK3 (GETA),,https://us02web.zoom.us/j/5403514055?pwd=ZnZ2d...,J000000030,fc-event-info,2023-07-11,2023-07-12,None,None,active,migrasi,None,0,None,2026-06-02 14:22:58.978733,2026-06-02 14:22:58.979526
3,08 SO 2A SelK3 (GETA),,https://us02web.zoom.us/j/5403514055?pwd=ZnZ2d...,J000000030,fc-event-info,2023-07-13,2023-07-14,None,None,active,migrasi,None,0,None,2026-06-02 14:22:58.978733,2026-06-02 14:22:58.979526
4,08 SO 2A SelK3 (GETA),,https://us02web.zoom.us/j/5403514055?pwd=ZnZ2d...,J000000030,fc-event-info,2023-07-18,2023-07-19,None,None,active,migrasi,None,0,None,2026-06-02 14:22:58.978733,2026-06-02 14:22:58.979526


In [11]:
# =========================================================
# 5. TRANSFORMASI TABEL: jadwal_pengajar (sumber: jadwal_pengajar)
# =========================================================
print("⚡ Melakukan transformasi tabel 'jadwal_pengajar'...")

# Ambil data dari tabel jadwal_pengajar lama
df_pengajar_lama = pd.read_sql("SELECT * FROM jadwal_pengajar", db_old)
print(f"  Data mentah: {len(df_pengajar_lama)} baris")

# Filter: hanya yang idjadwal-nya masuk dalam jadwal valid
valid_jadwal_ids = set(df_jadwal_clean['idjadwal'])  # sudah didefinisikan sebelumnya
df_pengajar_lama = df_pengajar_lama[df_pengajar_lama['idjadwal'].isin(valid_jadwal_ids)]
print(f"  Setelah filter idjadwal: {len(df_pengajar_lama)} baris")

# Buat dataframe untuk insert (tanpa kolom id_jadwal_pengajar)
df_jadwal_pengajar = pd.DataFrame()

# Kolom id_jadwal (masih old_id string)
df_jadwal_pengajar['id_jadwal'] = df_pengajar_lama['idjadwal'].astype(str)

# Kolom id_user: prioritas dari 'idusers', jika kosong coba 'idguru' (tapi 'idguru' mungkin juga ID user)
# Asumsikan 'idusers' yang utama
if 'idusers' in df_pengajar_lama.columns:
    df_jadwal_pengajar['id_user'] = df_pengajar_lama['idusers']
else:
    # fallback ke 'idguru' jika ada
    df_jadwal_pengajar['id_user'] = df_pengajar_lama.get('idguru', None)

# Bersihkan id_user: pastikan string, NaN jadi None
df_jadwal_pengajar['id_user'] = df_jadwal_pengajar['id_user'].astype(str).replace('nan', None).fillna(None)

# Hapus baris yang tidak punya id_user (karena FK)
df_jadwal_pengajar = df_jadwal_pengajar.dropna(subset=['id_user'])
print(f"  Setelah drop null id_user: {len(df_jadwal_pengajar)} baris")

# Kolom created_at, updated_at opsional
df_jadwal_pengajar['created_at'] = pd.Timestamp.now()
df_jadwal_pengajar['updated_at'] = pd.Timestamp.now()

print(f"✓ Tabel 'jadwal_pengajar' siap. Shape: {df_jadwal_pengajar.shape}")
print("   Kolom:", list(df_jadwal_pengajar.columns))

# Masukkan ke dictionary data_migrasi
fase_4_afrida['jadwal_pengajar'] = df_jadwal_pengajar

⚡ Melakukan transformasi tabel 'jadwal_pengajar'...
  Data mentah: 641 baris
  Setelah filter idjadwal: 640 baris
  Setelah drop null id_user: 640 baris
✓ Tabel 'jadwal_pengajar' siap. Shape: (640, 4)
   Kolom: ['id_jadwal', 'id_user', 'created_at', 'updated_at']


In [12]:
display(df_jadwal_pengajar.head())

,id_jadwal,id_user,created_at,updated_at
0,J000000025,U00019,2026-06-02 14:22:59.017738,2026-06-02 14:22:59.018074
1,J000000029,U00026,2026-06-02 14:22:59.017738,2026-06-02 14:22:59.018074
2,J000000031,U00035,2026-06-02 14:22:59.017738,2026-06-02 14:22:59.018074
3,J000000039,U00038,2026-06-02 14:22:59.017738,2026-06-02 14:22:59.018074
4,J000000043,U00019,2026-06-02 14:22:59.017738,2026-06-02 14:22:59.018074


In [13]:
# =========================================================
# 6. TRANSFORMASI TABEL: jadwal_siswa (sumber: jadwal_siswa)
# =========================================================
print("⚡ Melakukan transformasi tabel 'jadwal_siswa'...")

# Ambil data dari tabel jadwal_siswa lama
df_siswa_lama = pd.read_sql("SELECT * FROM jadwal_siswa", db_old)
print(f"  Data mentah: {len(df_siswa_lama)} baris")

# Filter: hanya yang idjadwal-nya masuk dalam jadwal valid
valid_jadwal_ids = set(df_jadwal_clean['idjadwal'])
df_siswa_lama = df_siswa_lama[df_siswa_lama['idjadwal'].isin(valid_jadwal_ids)]
print(f"  Setelah filter idjadwal: {len(df_siswa_lama)} baris")

# Buat dataframe untuk insert (tanpa id_jadwal_siswa)
df_jadwal_siswa = pd.DataFrame()

# Kolom wajib
df_jadwal_siswa['id_siswa'] = df_siswa_lama['idsiswa']
df_jadwal_siswa['id_jadwal'] = df_siswa_lama['idjadwal'].astype(str)

# Tanggal
df_jadwal_siswa['tanggal_mulai'] = pd.to_datetime(df_siswa_lama['tgl_mulai'], errors='coerce').dt.date
df_jadwal_siswa['tanggal_keluar'] = pd.to_datetime(df_siswa_lama['tgl_keluar'], errors='coerce').dt.date
df_jadwal_siswa['tanggal_aktif'] = pd.to_datetime(df_siswa_lama['tgl_aktif'], errors='coerce').dt.date

# Tambahan sesi (numeric) - perbaikan
df_jadwal_siswa['tambahan_sesi'] = (
    pd.to_numeric(df_siswa_lama['tambahan_sesi'], errors='coerce')
    .fillna(0)
    .astype(int)
)

# Tambahan keterangan
df_jadwal_siswa['tambahan_keterangan'] = df_siswa_lama['tambahan_ket'].fillna('').astype(str)

# Status keluar (integer 0/1) - perbaikan
df_jadwal_siswa['status_keluar'] = (
    pd.to_numeric(df_siswa_lama['is_keluar'], errors='coerce')
    .fillna(0)
    .astype(int)
)

# Kolom tambahan
df_jadwal_siswa['created_at'] = pd.Timestamp.now()
df_jadwal_siswa['updated_at'] = pd.Timestamp.now()
# Hapus baris yang tidak punya id_siswa (karena FK)
df_jadwal_siswa = df_jadwal_siswa.dropna(subset=['id_siswa'])
print(f"  Setelah drop null id_siswa: {len(df_jadwal_siswa)} baris")

print(f"✓ Tabel 'jadwal_siswa' siap. Shape: {df_jadwal_siswa.shape}")
print("   Kolom:", list(df_jadwal_siswa.columns))

# Masukkan ke dictionary data_migrasi
fase_4_afrida['jadwal_siswa'] = df_jadwal_siswa

⚡ Melakukan transformasi tabel 'jadwal_siswa'...
  Data mentah: 3905 baris
  Setelah filter idjadwal: 3903 baris
  Setelah drop null id_siswa: 3903 baris
✓ Tabel 'jadwal_siswa' siap. Shape: (3903, 10)
   Kolom: ['id_siswa', 'id_jadwal', 'tanggal_mulai', 'tanggal_keluar', 'tanggal_aktif', 'tambahan_sesi', 'tambahan_keterangan', 'status_keluar', 'created_at', 'updated_at']


In [14]:
display(df_jadwal_siswa.head())

,id_siswa,id_jadwal,tanggal_mulai,tanggal_keluar,tanggal_aktif,tambahan_sesi,tambahan_keterangan,status_keluar,created_at,updated_at
0,S0000362,J000000025,NaT,NaT,NaT,0,,0,2026-06-02 14:22:59.114909,2026-06-02 14:22:59.115503
1,S0000363,J000000025,NaT,NaT,NaT,0,,0,2026-06-02 14:22:59.114909,2026-06-02 14:22:59.115503
2,S0000085,J000000029,NaT,NaT,NaT,0,,0,2026-06-02 14:22:59.114909,2026-06-02 14:22:59.115503
3,S0000088,J000000029,NaT,NaT,NaT,0,,0,2026-06-02 14:22:59.114909,2026-06-02 14:22:59.115503
4,S0000114,J000000029,NaT,NaT,NaT,0,,0,2026-06-02 14:22:59.114909,2026-06-02 14:22:59.115503


In [15]:
# =========================================================
# 7. TRANSFORMASI TABEL: catatan_kelas (sumber: catatan_kelas)
# =========================================================
print("⚡ Melakukan transformasi tabel 'catatan_kelas'...")

df_catatan_lama = pd.read_sql("SELECT * FROM catatan_kelas", db_old)
print(f"  Data mentah: {len(df_catatan_lama)} baris")

# Filter hanya yang idjadwal dan idjadwaldetil valid (berada dalam jadwal_clean dan jadwal_detail nanti)
# Kita bisa filter idjadwal dulu
valid_jadwal_ids = set(df_jadwal_clean['idjadwal'])
df_catatan_lama = df_catatan_lama[df_catatan_lama['idjadwal'].isin(valid_jadwal_ids)]
print(f"  Setelah filter idjadwal: {len(df_catatan_lama)} baris")

# Buat dataframe untuk insert (tanpa id_ck)
df_catatan_kelas = pd.DataFrame()

# Simpan old_id untuk mapping nanti
df_catatan_kelas['old_id_ck'] = df_catatan_lama['idcatatan_kelas']

# Kolom FK yang perlu mapping
df_catatan_kelas['id_jadwal'] = df_catatan_lama['idjadwal'].astype(str)   # old jadwal id
df_catatan_kelas['id_jadwal_detail'] = df_catatan_lama['idjadwaldetil']   # old detail id, nanti di-map

# Kolom data
df_catatan_kelas['catatan_kelas'] = df_catatan_lama['catatan'].fillna('').astype(str)
df_catatan_kelas['topik_diskusi'] = df_catatan_lama['materi_diskusi'].fillna('').astype(str)
df_catatan_kelas['tanggal_konfirmasi'] = pd.to_datetime(df_catatan_lama['tglcek'], errors='coerce')
df_catatan_kelas['hasil_konfirmasi'] = df_catatan_lama['hasil_konfirm'].fillna('').astype(str)

# Kolom created_at, updated_at opsional
df_catatan_kelas['created_at'] = pd.Timestamp.now()
df_catatan_kelas['updated_at'] = pd.Timestamp.now()

print(f"✓ Tabel 'catatan_kelas' siap. Shape: {df_catatan_kelas.shape}")
print("   Kolom:", list(df_catatan_kelas.columns))

⚡ Melakukan transformasi tabel 'catatan_kelas'...
  Data mentah: 12797 baris
  Setelah filter idjadwal: 12797 baris
✓ Tabel 'catatan_kelas' siap. Shape: (12797, 9)
   Kolom: ['old_id_ck', 'id_jadwal', 'id_jadwal_detail', 'catatan_kelas', 'topik_diskusi', 'tanggal_konfirmasi', 'hasil_konfirmasi', 'created_at', 'updated_at']


In [16]:
# =========================================================
# 8. TRANSFORMASI TABEL: catatan_kelas_tag (sumber: catatan_kelas_tag)
# =========================================================
print("⚡ Melakukan transformasi tabel 'catatan_kelas_tag'...")

df_tag_lama = pd.read_sql("SELECT * FROM catatan_kelas_tag", db_old)
print(f"  Data mentah: {len(df_tag_lama)} baris")

# Filter hanya yang idcatatan_kelas-nya masih ada di df_catatan_kelas (old_id)
valid_old_ck_ids = set(df_catatan_kelas['old_id_ck'])
df_tag_lama = df_tag_lama[df_tag_lama['idcatatan_kelas'].isin(valid_old_ck_ids)]
print(f"  Setelah filter idcatatan_kelas: {len(df_tag_lama)} baris")

df_catatan_kelas_tag = pd.DataFrame()

# Simpan old_id_ck untuk mapping nanti
df_catatan_kelas_tag['old_id_ck'] = df_tag_lama['idcatatan_kelas']

# Mapping: idtagmd -> id_topik_diskusi (asumsi langsung)
df_catatan_kelas_tag['id_topik_diskusi'] = df_tag_lama['idtagmd']

# Kolom id_ck_tag akan auto increment, tidak disertakan

print(f"✓ Tabel 'catatan_kelas_tag' siap. Shape: {df_catatan_kelas_tag.shape}")
print("   Kolom:", list(df_catatan_kelas_tag.columns))


⚡ Melakukan transformasi tabel 'catatan_kelas_tag'...
  Data mentah: 999 baris
  Setelah filter idcatatan_kelas: 999 baris
✓ Tabel 'catatan_kelas_tag' siap. Shape: (999, 2)
   Kolom: ['old_id_ck', 'id_topik_diskusi']


In [17]:
# =========================================================
# 7. TRANSFORMASI TABEL: catatan_mingguan
# =========================================================
print("⚡ Melakukan transformasi tabel 'catatan_mingguan'...")

df_cm_lama = pd.read_sql("SELECT * FROM catatan_mingguan", db_old)
print(f"  Data mentah: {len(df_cm_lama)} baris")

# Filter jika perlu (misal berdasarkan id_jadwal? catatan_mingguan tidak punya id_jadwal langsung, mungkin idusers? Tidak difilter)

df_catatan_mingguan = pd.DataFrame()

# Kolom FK
df_catatan_mingguan['id_user'] = df_cm_lama['idusers'].astype(str)

# Tanggal
df_catatan_mingguan['tanggal_mulai_cm'] = pd.to_datetime(df_cm_lama['tglawal'], errors='coerce').dt.date
df_catatan_mingguan['tanggal_selesai_cm'] = pd.to_datetime(df_cm_lama['tglakhir'], errors='coerce').dt.date
df_catatan_mingguan['tanggal_verifikasi_cm'] = pd.to_datetime(df_cm_lama['tglcek'], errors='coerce')

# Teks
df_catatan_mingguan['keterangan_cm'] = df_cm_lama['catatan'].fillna('').astype(str)
df_catatan_mingguan['keputusan_cm'] = df_cm_lama['hasil_konfirmasi'].fillna('').astype(str)

# Kolom tambahan
df_catatan_mingguan['created_at'] = pd.Timestamp.now()
df_catatan_mingguan['updated_at'] = pd.Timestamp.now()

# Hapus baris yang tidak punya id_user (FK)
before = len(df_catatan_mingguan)
df_catatan_mingguan = df_catatan_mingguan.dropna(subset=['id_user'])
print(f"  Drop {before - len(df_catatan_mingguan)} baris karena id_user kosong")

print(f"✓ Tabel 'catatan_mingguan' siap. Shape: {df_catatan_mingguan.shape}")
print("   Kolom:", list(df_catatan_mingguan.columns))

⚡ Melakukan transformasi tabel 'catatan_mingguan'...
  Data mentah: 0 baris
  Drop 0 baris karena id_user kosong
✓ Tabel 'catatan_mingguan' siap. Shape: (0, 8)
   Kolom: ['id_user', 'tanggal_mulai_cm', 'tanggal_selesai_cm', 'tanggal_verifikasi_cm', 'keterangan_cm', 'keputusan_cm', 'created_at', 'updated_at']


In [18]:
data_migrasi = {
    'jadwal': df_jadwal_insert,          # dataframe tanpa id_jadwal, tapi punya old_id di list terpisah
    'jadwal_old_ids': old_id_list,       # list old id jadwal (string)
    'jadwal_hari': df_jadwal_hari,       # kolom id_jadwal berisi string old
    'jadwal_detail': df_jadwal_detail_insert,   # punya kolom id_jadwal_detail (old) dan id_jadwal (old string)
    'jadwal_detail_old_ids': old_detail_ids,  # kita perlu buat ini sebelumnya! 
    'jadwal_pengajar': df_jadwal_pengajar,
    'jadwal_siswa': df_jadwal_siswa,
    'catatan_kelas': df_catatan_kelas,   # punya old_id_ck, id_jadwal(old), id_jadwal_detail(old)
    'catatan_kelas_tag': df_catatan_kelas_tag,  # punya old_id_ck, id_topik_diskusi
    'catatan_mingguan': df_catatan_mingguan,
}

# Simpan ke pickle
with open('fase_4_afrida.pkl', 'wb') as f:
    pickle.dump(data_migrasi, f)

## 3. Transform Data (jika diperlukan)

## 4. Insert ke DB Baru

## 5. Verifikasi Data

## 6. Return Hasil Migrasi untuk migrate_db.py

## 7. Close Connection